In [6]:
import pandas as pd
from pathlib import Path

# Fraction of nulls above which a column is flagged in the report and dropped by the prune.
NULL_FLAG_THRESHOLD = 0.50

# name -> (source file, pruned output file)
DATASETS = {
    "listings": ("combined_listings.csv", "combined_listingsv2.csv"),
    "sold": ("combined_sold.csv", "combined_soldv2.csv"),
}

# scripts/tools -> scripts -> repo root
base_dir = Path.cwd().parent.parent
processed_path = base_dir / "data" / "processed"
processed_path

PosixPath('/Users/alexanderchin/code/idx-data-analysis/data/processed')

## 1. Missing value report

Flags every column whose null rate exceeds `NULL_FLAG_THRESHOLD` and returns them (highest null rate first).

In [7]:
def missing_value_report(df, threshold=NULL_FLAG_THRESHOLD):
    null_pct = df.isnull().mean().sort_values(ascending=False)
    flagged = null_pct[null_pct > threshold]

    if flagged.empty:
        print(f"No columns exceed {threshold:.0%} null values.")
    else:
        print(f"{len(flagged)} column(s) flagged:\n")
        print(f"{'column':<32}{'null_pct':>10}")
        print("-" * 42)
        for col, pct in flagged.items():
            print(f"{col:<32}{pct:>10.1%}")

    return flagged

In [ ]:
FRAMES = {}
NULL_REPORTS = {}

for name, (source_name, _) in DATASETS.items():
    source = processed_path / source_name
    print("=" * 60)
    print(f"[{name}] Reading source: {source}")
    print("=" * 60)
    FRAMES[name] = pd.read_csv(source, low_memory=False)
    NULL_REPORTS[name] = missing_value_report(FRAMES[name])
    print()

[listings] Reading source: /Users/alexanderchin/code/idx-data-analysis/data/processed/combined_listings.csv
33 column(s) flagged:

column                            null_pct
------------------------------------------
FireplacesTotal                     100.0%
ElementarySchoolDistrict            100.0%
CoveredSpaces                       100.0%
TaxYear                             100.0%
BusinessType                        100.0%
TaxAnnualAmount                     100.0%
AboveGradeFinishedArea              100.0%
MiddleOrJuniorSchoolDistrict        100.0%
BelowGradeFinishedArea               99.4%
CoBuyerAgentFirstName                97.8%
BuilderName                          95.4%
LotSizeDimensions                    94.8%
BuyerAgencyCompensation              91.7%
BuyerAgencyCompensationType          91.6%
BuildingAreaTotal                    91.0%
MiddleOrJuniorSchool                 88.1%
ElementarySchool                     88.1%
HighSchool                           84.4%
ClosePric

## 2. Prune columns

Drops the columns flagged in part 1 and writes the pruned `*v2.csv`. This reuses reports cached in `FRAMES` / `NULL_REPORTS`

In [11]:
def prune_dataset(df, flagged, output_name):
    output = processed_path / output_name

    to_drop = flagged.index.tolist()
    pruned = df.drop(columns=to_drop)

    print("=" * 60)
    print(f"Dropped (> {NULL_FLAG_THRESHOLD:.0%} null): {len(to_drop)} column(s)")
    print(f"Columns: {df.shape[1]} -> {pruned.shape[1]}")
    print("=" * 60)

    pruned.to_csv(output, index=False)
    print(f"Wrote {len(pruned):,} rows x {pruned.shape[1]} cols to:")
    print(f"  {output}")

    return pruned

In [12]:
for name, (_, output_name) in DATASETS.items():
    print(f"\n### {name} ###")
    prune_dataset(FRAMES[name], NULL_REPORTS[name], output_name)


### listings ###
Dropped (> 50% null): 33 column(s)
Columns: 84 -> 51
Wrote 533,052 rows x 51 cols to:
  /Users/alexanderchin/code/idx-data-analysis/data/processed/combined_listingsv2.csv

### sold ###
Dropped (> 50% null): 29 column(s)
Columns: 84 -> 55
Wrote 458,336 rows x 55 cols to:
  /Users/alexanderchin/code/idx-data-analysis/data/processed/combined_soldv2.csv
